# Phase 6: Retrieval Pipeline

Evaluate Dense, Sparse (BM25), and Hybrid (RRF) retrieval methods across 3 chunking strategies.
Uses `qa_pairs_filtered.parquet` for evaluation.

In [1]:
import os, sys, subprocess
from pathlib import Path

# Detect environment
def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    REPO_ROOT = Path('/content/rag-vn-finance')
    if not REPO_ROOT.exists():
        print("Đang tải mã nguồn và cài đặt thư viện lần đầu...")
        subprocess.run(['git', 'clone', 'https://github.com/thong7d/rag-vn-finance.git', str(REPO_ROOT)])
        
        req_path = REPO_ROOT / 'requirements.txt'
        if req_path.exists():
            os.system(f'pip install -r "{req_path}" -q')
            
        print("Cài đặt hoàn tất. Đang tự động khởi động lại Kernel để nạp thư viện lõi (Numpy/Torch)...")
        os.kill(os.getpid(), 9) # Tự động ngắt tiến trình để ép Colab khởi động lại RAM
    else:
        print("Mã nguồn đã tồn tại. Bỏ qua cài đặt...")
else:
    REPO_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())

print(f"Project root: {REPO_ROOT}")
assert REPO_ROOT.exists(), f"Project root not found: {REPO_ROOT}"

src_path = str(REPO_ROOT)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Load file .env
from dotenv import load_dotenv
if IN_COLAB:
    load_dotenv('/content/drive/MyDrive/rag-vn-finance/.env') 
else:
    load_dotenv(REPO_ROOT / '.env')

print("\nColab setup complete.")

Environment: Local
Project root: d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\implementation

Colab setup complete.


## 1. Environment Setup & Model Loading
Import necessary libraries, load project configurations from `config.yaml`, and initialize the SentenceTransformer model used for encoding user queries.

In [2]:
import json
import pandas as pd
import faiss
from tqdm import tqdm  # Đã đổi từ tqdm.notebook sang tqdm chuẩn để tránh lỗi hiển thị trên Colab
from sentence_transformers import SentenceTransformer
import torch

from src.utils import load_config, resolve_path, ensure_dir, get_env
from src.indexing import load_bm25_index
from src.retrieval import DenseRetriever, SparseRetriever, HybridRetriever, calculate_metrics, create_dense_retriever

# Load config
config = load_config()

# Load Model
model_name = config['embedding']['model_name']
device = config['embedding'].get('device', 'cpu')
if device == 'auto':
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer(model_name, device=device)
print(f"Loaded {model_name} on {model.device}")

d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\.venv\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loaded intfloat/multilingual-e5-large on cpu


## 2. Load Evaluation Data
Load the synthetic QA pairs generated in Phase 5. We use the full `qa_pairs_filtered.parquet` dataset to evaluate our retrieval methods.

In [3]:
qa_dir = resolve_path(config['synthetic_qa'], 'output_dir')
# Sử dụng REPO_ROOT để tạo đường dẫn tuyệt đối chính xác
qa_path = os.path.join(REPO_ROOT, qa_dir, 'ground_truth_final.jsonl')

if not os.path.exists(qa_path):
    raise FileNotFoundError(f"{qa_path} not found. Ensure Phase 5 is completed.")

# Đọc file JSONL
qa_list = []
with open(qa_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            qa_list.append(json.loads(line))

df_qa = pd.DataFrame(qa_list)
print(f"Loaded {len(df_qa)} QA pairs for evaluation from ground_truth_final.jsonl.")


Loaded 150 QA pairs for evaluation from ground_truth_final.jsonl.


## 3. Retrieval Evaluation Loop
Iterate through all 3 chunking strategies (`fixed_size`, `sentence_aware`, `article_level`). For each strategy:
1. Load the corresponding FAISS and BM25 indices.
2. Instantiate Dense, Sparse, and Hybrid retrievers.
3. Run all evaluation queries and compute `Precision@K`, `Recall@K`, `MRR`, and `NDCG@10`.
4. Aggregate the metrics.

In [4]:
strategies = config['chunking']['strategies']
top_k_dense = config['retrieval']['top_k_dense']
top_k_sparse = config['retrieval']['top_k_sparse']
top_k_hybrid = config['retrieval']['top_k_hybrid']
rrf_k = config['retrieval']['rrf_k']

index_base_dir = os.path.join(REPO_ROOT, resolve_path(config['indexing'], 'output_dir'))
bm25_base_dir = os.path.join(REPO_ROOT, resolve_path(config['indexing'], 'bm25_dir'))

results = []

for strategy in strategies:
    print(f"\n{'='*40}\nEvaluating Strategy: {strategy}\n{'='*40}")
    
    # --- Xác định Backend & Nạp Chunk IDs ---
    backend = os.environ.get("VECTOR_STORE_BACKEND", "faiss").lower()
    chunk_ids_path = os.path.join(index_base_dir, strategy, "chunk_ids.json")
    
    if not os.path.exists(chunk_ids_path):
        print(f"Missing chunk_ids.json for {strategy}. Skipping.")
        continue
        
    with open(chunk_ids_path, 'r', encoding='utf-8') as f:
        dense_chunk_ids = json.load(f)
        
    # --- Khởi tạo Dense Retriever ---
    if backend == "faiss":
        faiss_path = os.path.join(index_base_dir, strategy, "index.faiss")
        if not os.path.exists(faiss_path):
            print(f"Missing Dense index for {strategy}. Skipping.")
            continue
        faiss_index = faiss.read_index(faiss_path)
        dense_retriever = create_dense_retriever(
            backend="faiss",
            index=faiss_index,
            chunk_ids=dense_chunk_ids,
            model=model
        )
    elif backend == "qdrant":
        from qdrant_client import QdrantClient
        collection_name = f"{config.get('vector_store', {}).get('qdrant', {}).get('collection_name', 'vn_finance')}_{strategy}"
        qdrant_url = get_env("QDRANT_URL")
        qdrant_api_key = get_env("QDRANT_API_KEY")
        
        if qdrant_url:
            qdrant_client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)
        else:
            local_path = config.get("vector_store", {}).get("qdrant", {}).get("local_path", "qdrant_data")
            qdrant_dir = os.path.join(REPO_ROOT, local_path, strategy)
            if not os.path.exists(qdrant_dir):
                print(f"Missing Qdrant index folder for {strategy} at {qdrant_dir}. Skipping.")
                continue
            qdrant_client = QdrantClient(path=qdrant_dir)
            
        dense_retriever = create_dense_retriever(
            backend="qdrant",
            client=qdrant_client,
            collection_name=collection_name,
            chunk_ids=dense_chunk_ids,
            model=model
        )
        
    # --- Load Sparse Index ---
    try:
        bm25_index, sparse_chunk_ids = load_bm25_index(bm25_base_dir, strategy)
    except FileNotFoundError:
        print(f"Missing Sparse index for {strategy}. Skipping.")
        continue
        
    sparse_retriever = SparseRetriever(bm25_index, sparse_chunk_ids)
    
    # --- Hybrid Retriever ---
    hybrid_retriever = HybridRetriever(dense_retriever, sparse_retriever, rrf_k=rrf_k)
    
    # --- Evaluate ---
    strategy_metrics = {'Dense': [], 'Sparse': [], 'Hybrid': []}
    
    print("Bắt đầu chạy vòng lặp đánh giá (vui lòng đợi vài phút cho mỗi strategy)...")
    
    # Progress bar over the evaluation dataset
    for _, row in tqdm(df_qa.iterrows(), total=len(df_qa), desc=f"Evaluating {strategy}"):
        query = row['question']
        ground_truth_doc_id = row['doc_id']
        
        # Dense
        dense_res = dense_retriever.retrieve(query, top_k=top_k_dense)
        dense_ids = [cid for cid, _ in dense_res]
        strategy_metrics['Dense'].append(calculate_metrics(dense_ids, ground_truth_doc_id, k=top_k_dense))
        
        # Sparse
        sparse_res = sparse_retriever.retrieve(query, top_k=top_k_sparse)
        sparse_ids = [cid for cid, _ in sparse_res]
        strategy_metrics['Sparse'].append(calculate_metrics(sparse_ids, ground_truth_doc_id, k=top_k_sparse))
        
        # Hybrid
        hybrid_res = hybrid_retriever.retrieve(query, top_k=top_k_hybrid)
        hybrid_ids = [cid for cid, _ in hybrid_res]
        strategy_metrics['Hybrid'].append(calculate_metrics(hybrid_ids, ground_truth_doc_id, k=top_k_hybrid))
        
    # --- Aggregate and Store Results ---
    for method, metrics_list in strategy_metrics.items():
        df_m = pd.DataFrame(metrics_list)
        avg_metrics = df_m.mean().to_dict()
        
        res_row = {
            'Strategy': strategy,
            'Method': method
        }
        res_row.update(avg_metrics)
        results.append(res_row)
        
        # Format metrics for printing
        metrics_str = ", ".join([f"{k}: {v:.4f}" for k, v in avg_metrics.items()])
        print(f"{method:10} -> {metrics_str}")
        
    # --- Checkpoint after each strategy ---
    df_results = pd.DataFrame(results)
    eval_dir = os.path.join(REPO_ROOT, resolve_path(config['evaluation'], 'output_dir'))
    ensure_dir(eval_dir)
    out_path = os.path.join(eval_dir, f"retrieval_benchmark_{backend}.csv")
    df_results.to_csv(out_path, index=False)
    print(f"\n[CHECKPOINT] Đã lưu kết quả của {strategy} vào {out_path}")


Evaluating Strategy: fixed_size


C:\Users\Inspiron\AppData\Local\Temp\ipykernel_3328\237628896.py:53: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <vn_finance_fixed_size> contains 45764 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant_client = QdrantClient(path=qdrant_dir)
[2026-07-06 18:19:59] [INFO] src.indexing: [fixed_size] BM25 index loaded — 45,764 chunks, avgdl=206.0


Bắt đầu chạy vòng lặp đánh giá (vui lòng đợi vài phút cho mỗi strategy)...


Evaluating fixed_size: 100%|██████████| 150/150 [10:04<00:00,  4.03s/it]


Dense      -> Precision@10: 0.2800, Recall@10: 0.9000, MRR: 0.7555, NDCG@10: 0.7696
Sparse     -> Precision@10: 0.2347, Recall@10: 0.9200, MRR: 0.7882, NDCG@10: 0.7912
Hybrid     -> Precision@10: 0.2800, Recall@10: 0.9267, MRR: 0.8191, NDCG@10: 0.8103

[CHECKPOINT] Đã lưu kết quả của fixed_size vào d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\implementation\evaluation\retrieval_benchmark_qdrant.csv

Evaluating Strategy: sentence_aware


...

Dense      -> Precision@10: 0.2707, Recall@10: 0.9267, MRR: 0.7571, NDCG@10: 0.7772
Sparse     -> Precision@10: 0.2187, Recall@10: 0.8733, MRR: 0.7705, NDCG@10: 0.7620
Hybrid     -> Precision@10: 0.2700, Recall@10: 0.9600, MRR: 0.8196, NDCG@10: 0.8186

[CHECKPOINT] Đã lưu kết quả của sentence_aware vào d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\implementation\evaluation\retrieval_benchmark_qdrant.csv

Evaluating Strategy: article_level


[2026-07-06 18:43:14] [INFO] src.indexing: [article_level] BM25 index loaded — 9,999 chunks, avgdl=404.6


Bắt đầu chạy vòng lặp đánh giá (vui lòng đợi vài phút cho mỗi strategy)...


Evaluating article_level: 100%|██████████| 150/150 [06:18<00:00,  2.52s/it]

Dense      -> Precision@10: 0.0933, Recall@10: 0.9333, MRR: 0.7847, NDCG@10: 0.8206
Sparse     -> Precision@10: 0.0920, Recall@10: 0.9200, MRR: 0.7960, NDCG@10: 0.8261
Hybrid     -> Precision@10: 0.0953, Recall@10: 0.9533, MRR: 0.8193, NDCG@10: 0.8521

[CHECKPOINT] Đã lưu kết quả của article_level vào d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\implementation\evaluation\retrieval_benchmark_qdrant.csv


## 4. Save Benchmark Results
Export the final evaluation results to `evaluation/retrieval_benchmark.csv` for use in subsequent phases or reporting.

In [5]:
if results:
    backend = os.environ.get("VECTOR_STORE_BACKEND", "faiss").lower()
    df_results = pd.DataFrame(results)
    eval_dir = os.path.join(REPO_ROOT, resolve_path(config['evaluation'], 'output_dir'))
    ensure_dir(eval_dir)
    
    out_path = os.path.join(eval_dir, f"retrieval_benchmark_{backend}.csv")
    df_results.to_csv(out_path, index=False)
    print(f"\nHoàn thành! Đã lưu kết quả cuối cùng vào {out_path}")
    display(df_results)
else:
    print("No results to save. Ensure indexes and QA data are present.")


Hoàn thành! Đã lưu kết quả cuối cùng vào d:\000MINHTHONG\Junior - Semester II\TDM & A\FinalProject\finance-news\implementation\evaluation\retrieval_benchmark_qdrant.csv


,Strategy,Method,Precision@10,Recall@10,MRR,NDCG@10
0,fixed_size,Dense,0.280000,0.900000,0.755545,0.769594
1,fixed_size,Sparse,0.234667,0.920000,0.788201,0.791224
2,fixed_size,Hybrid,0.280000,0.926667,0.819056,0.810335
3,sentence_aware,Dense,0.270667,0.926667,0.757101,0.777243
4,sentence_aware,Sparse,0.218667,0.873333,0.770526,0.762027
5,sentence_aware,Hybrid,0.270000,0.960000,0.819619,0.818638
6,article_level,Dense,0.093333,0.933333,0.784701,0.820574
7,article_level,Sparse,0.092000,0.920000,0.795979,0.826125
8,article_level,Hybrid,0.095333,0.953333,0.819304,0.852113
